# LLMFactor: Extracting Profitable Factors through Prompts for Explainable Stock Movement Prediction
# Dataset: stocknet

1. **Skeleton Stage**：拆解問題的主要框架或骨架。
2. **Point-expanding Stage**：進一步擴展每個骨架的具體細節。
將前一日的新聞資料依據 Skeleton Stage 拆解的每個判斷依據都做一次判斷隔日漲跌  
e.g. 20230104的新聞判斷漲跌後存在20230105  
儲存格式：  
20230105  
判斷依據1: 由20230104的新聞判斷為漲  
判斷依據2: 由20230104的新聞判斷為跌  
...  


檔案說明：  
存在 


In [12]:
import os
import json
import textwrap
import pandas as pd
import numpy as np
from collections import Counter
from datetime import datetime
from dateutil.relativedelta import relativedelta
from IPython.display import display
from IPython.display import Markdown

from langchain_openai import ChatOpenAI


def to_markdown(text):
  text = text.replace('•', '  *')
  return Markdown(textwrap.indent(text, '> ', predicate=lambda _: True))

llm = ChatOpenAI(
  openai_api_key = os.getenv('OPENAI_API_KEY'),
  model='gpt-4o-mini',
  temperature=1,
)

# print(llm.invoke("hello world"))
# print(llm.model_name)
# print(llm.temperature)

# path_price = os.path.join(os.path.abspath(os.path.dirname(os.getcwd())), 'history_data', 'tw', 'stock_price', id + 'tech.csv')
# path_news_title = os.path.dirname(os.path.abspath(os.getcwd())) + "/history_data/tw/news_title/" + id + "news_title.json"
# stock_id = "2454"

# Data

In [13]:
symbol = "AAPL"
path_price = os.path.dirname(os.path.abspath(os.getcwd())) + "/history_data/us/dataset/stocknet/price/raw/" + symbol
path_news = os.path.dirname(os.path.abspath(os.getcwd())) + "/history_data/us/dataset/stocknet/tweet/preprocessed/" + symbol
path_out = "out_llmfactor_stocknet/" + symbol

# 寫一個從第2014年開始的函數
def get_date_range(start_date, end_date):
    start_date = datetime.strptime(start_date, "%Y-%m-%d")
    end_date = datetime.strptime(end_date, "%Y-%m-%d")
    date_list = []
    while start_date <= end_date:
        date_list.append(start_date.strftime("%Y-%m-%d"))
        start_date += relativedelta(days=1)
    return date_list

date_list = get_date_range("2014-01-01", "2015-12-30")
# print(date_list)

# # 讀取新聞
# for day in date_list:
#     if os.path.exists(path_news + "/" + day) == False:
#         continue
#     with open(path_news + "/" + day, 'r') as f:
#         news = f.read()
#         print(news)

# 讀取股價
price = pd.read_csv(path_price + ".csv")

# 抓特定時間範圍的股價
price = price[price['Date'].isin(date_list)]
print(price)
# 計算股價變化
price["Close_Change"] = price["Close"].pct_change()
price["Movement"] = np.where(price["Close_Change"] > 0, 1, 0)
print(price)
actual_date_list = price["Date"].tolist()
print(actual_date_list)

           Date        Open        High         Low       Close   Adj Close  \
333  2014-01-02   79.382858   79.575714   78.860001   79.018570   73.522530   
334  2014-01-03   78.980003   79.099998   77.204285   77.282860   71.907555   
335  2014-01-06   76.778572   78.114288   76.228569   77.704285   72.299644   
336  2014-01-07   77.760002   77.994286   76.845711   77.148575   71.782608   
337  2014-01-08   76.972855   77.937141   76.955711   77.637146   72.237190   
..          ...         ...         ...         ...         ...         ...   
831  2015-12-23  107.269997  108.849998  107.199997  108.610001  104.940506   
832  2015-12-24  109.000000  109.000000  107.949997  108.029999  104.380112   
833  2015-12-28  107.589996  107.690002  106.180000  106.820000  103.210999   
834  2015-12-29  106.959999  109.430000  106.860001  108.739998  105.066116   
835  2015-12-30  108.580002  108.699997  107.180000  107.320000  103.694107   

        Volume  
333   58671200  
334   98116900  


Task Definition
Input: stock's date, stock's news, past stock price movements
* Sequential Knowledge-Guided Prompting
Step 1 Matching and Acquiring News Background Knowledge
Ticker
Industry
AAPL Computer Manufacturing.match
MSFT Computer Software
TESLA TSLA Auto Manufacturing
<Stock List>
NEWS Stockmatch-
1
<stock target = NVDA>
Step 2 Generating Factors that May Affect Stock Prices
Output: today's stock price movement
Factor Analysis in the Stock Market
Please fill in the blank: stock target and
stockmatch are most likely in a relationship
stock target and stockmatch are most likely
in a competitive relationship


Please extract the top k factors that may affect the stock price of stock target from the following news:
Recent News - Nvidia stock has gained more than 20% in January alone. Is it a buy right now? ...
a) Nvidia stock gain in January b) New product announcements c) Selection of Nvidia Drive Thor by EV makers



Step 3 Predicting Stock Price Movements
B
----------------
Based on the following information, please judge the direction of the stock price from rise/fall, fill in the blank
and give reasons: Relations+Factors +Past stock price movements
On Feb. 1, the stock price of stocktarget will
Rise. Based on the information provided and patterns for the previous trading days...
guide
APPLICATION
•1 0
6469
Please tell me the reason why the stock price<
went up on this day
a) Nvidia stock gain in January
b) New product announcements
c) Selection of Nvidia Drive Thor by EV makers

In [7]:
import json
import pandas as pd
import re
from datetime import datetime

def get_symbol_in_news(news_data):
    #news_data: Sample news data as multi-line JSON (each line is a JSON object)
    
    path_stocktable =os.path.dirname(os.path.abspath(os.getcwd())) + "/history_data/us/dataset/stocknet/StockTable"

    with open(path_stocktable, 'r') as f:
        stocktable_data = f.read()

    from io import StringIO
    stocktable_df = pd.read_csv(StringIO(stocktable_data), delimiter="\t")

    # Convert stocktable into a dictionary for easy lookup by Symbol
    stock_lookup = {row['Symbol']: (row['Sector'], row['Company']) for _, row in stocktable_df.iterrows()}

    # Split the news data into lines
    news_lines = news_data.strip().splitlines()

    # Prepare to store matching news entries with stock data
    matched_news = []

    # Define a regex to match symbols with "$" prefix
    symbol_regex = re.compile(r"\$\s*(\w+|\w+-\w+)")

    # Process each news entry
    for line in news_lines:
        # news_entry = json.loads(line)
        # news_text = news_entry["text"]
        # created_at = news_entry["created_at"]
        matched_stocks = []
        
        # Find all stock symbols in the news text
        # 忽略大小寫
        symbols = set(symbol_regex.findall(line))
        
        for symbol in symbols:
            # print(symbol)
            # Reformat to match stock symbol format in stocktable (e.g., "$AAPL")
            symbol_with_dollar = f"${symbol.upper()}"
            
            # Check if symbol exists in stocktable
            if symbol_with_dollar in stock_lookup:
                sector, company = stock_lookup[symbol_with_dollar]
                matched_stocks.append({
                    "Sector": sector,
                    "Symbol": symbol_with_dollar,
                    "Company": company,
                    # "News Date": created_at,
                    # "News Text": " ".join(news_text)
                })
        
        # If there are matched stocks, add them to the final list
        if matched_stocks:
            matched_news.extend(matched_stocks)

    # Convert matched news to a DataFrame for easier viewing
    matched_news_df = pd.DataFrame(matched_news)
    # 移除重複的股票
    matched_news_df = matched_news_df.drop_duplicates(subset=['Symbol'])

    return matched_news_df

In [ ]:
def get_relation(stock_symbol, news):
    stock_matchs = get_symbol_in_news(news)
    text_relation = ""

    for index, row in stock_matchs.iterrows():
        stock_match = row["Symbol"].replace("$", "")
        if stock_symbol == stock_match:
            continue
        # print(row["Symbol"])
        # print(row)

        messages = [
        ("human",
            f"""Please fill in the blank and return a complete sentence: {stock_symbol} and {stock_match} are most likely in a _______ relationship.""")
        ]

        res = llm.invoke(messages)
        text_relation += res.content + "\n"

    return text_relation


def llm_predict(llm, stock_symbo, list_date, list_move, news, k=3):

    # step 1
    relation = get_relation(stock_symbo, news)
    print("Relation:")
    print(relation)

    # step 2 
    message_factor = [
    ("human",
        f"""Please extract the top {k} factors that may affect the stock price of {stock_symbo} from the following news: {news}""")
    ]

    res = llm.invoke(message_factor)
    factor = res.content
    print("Factor:")
    print(factor)

    # step 3 : Predicting Stock Price Movements
    messages_movement = [
    ("human",
        f"""
        Based on the following information, please judge the direction of the stock price from rise/fall, fill in the blank and give reasons. These are the main factors that may affect this stock’s price recently: {factor}.
        These are the connections between the companies that have appeared in the news: {relation}.
        On {list_date[0]}, the stock price of {stock_symbo} {list_move[0]}.
        On {list_date[1]}, the stock price of {stock_symbo} {list_move[1]}.
        On {list_date[2]}, the stock price of {stock_symbo} {list_move[2]}.
        On {list_date[3]}, the stock price of {stock_symbo} {list_move[3]}.
        On {list_date[4]}, the stock price of {stock_symbo} {list_move[4]}.
        On {list_date[5]}, the stock price of {stock_symbo} will _______ .
        """)
    ]
    # print(messages_movement)

    response_movement = llm.invoke(messages_movement)
    print("Movement:")
    print(response_movement.content)

    return response_movement.content

stock_symbo = "AAPL"
dict_res = {}
for i in range(6, len(actual_date_list)):
    print("日期", actual_date_list[i-1])

    # 處理新聞
    news_data = ""
    if os.path.exists(path_news + "/" + actual_date_list[i-2]) == False:
        continue
    with open(path_news + "/" + actual_date_list[i-2], 'r') as f:
        news = f.read()
        # Split the news data into lines
        news_lines = news.strip().splitlines()
        # Process each news entry
        for line in news_lines:
            news_entry = json.loads(line)
            news_text = news_entry["text"]
            news_data += " ".join(news_text)
            news_data += "\n"
    # out = get_symbol_in_news(news_data)
    # print(out)
    
    
    # 預測
    date_list_tmp = actual_date_list[i-6:i]
    movement_list = []
    for d in date_list_tmp:
        t = ""
        if price[price['Date'] == d]["Movement"].values == 1:
            t = "rose"
        else:
            t = "fell"
        movement_list.append(t)
    
    # print(date_list_tmp)
    # print(movement_list)
    print()

    res_movement = llm_predict(llm, stock_symbo, date_list_tmp, movement_list, news_data)

    # 建立一個字典存放結果
    dict_res[actual_date_list[i-1]] = res_movement

    print(dict_res)
    os.makedirs(os.path.dirname(path_out + "2.json"), exist_ok=True)
    with open(path_out + "1.json", 'w') as f:
        json.dump(dict_res, f, indent=4)



# Eval

In [18]:
def eval(df_price, data):
    # print(data)
    # 計算準確率和平均報酬率
    gain = []
    loss = []
    gain_precision = []
    loss_precision = []

    tp = 0
    fp = 0
    tn = 0
    fn = 0

    ttl_count = 0

    for date_str, value in data.items():
        ttl_count += 1
        # print(date_str)
        # print(value)

        sig = 0
        
        # 抓到在value裡面的Resason字出現之前的字串，無視大小寫
        # Extract everything before the word "Reasons" (case-insensitive)
        judge = re.split(r'reasons', value, flags=re.IGNORECASE)[0]
        # print(judge.strip())  # Output: "This is a test string."

        if re.search(r"rise", judge, flags=re.IGNORECASE):
            sig = 1
        elif re.search(r"fall", judge, flags=re.IGNORECASE):
            sig = -1

        # if re.search(r"fall", judge, flags=re.IGNORECASE):
        #     sig = -1
        # elif re.search(r"rise", judge, flags=re.IGNORECASE):
        #     sig = 1

        if df_price.empty:
            continue
        
        rtn = df_price[df_price['Date'] == date_str]["Close_Change"].values[0]
        # print(rtn)

        # if pd.isna(rtn):
        #     rtn = 0

        if sig > 0:
            if rtn > 0:
                tp += 1
                gain.append(rtn)
                gain_precision.append(rtn)
            else:
                fp += 1
                loss.append(rtn)
                loss_precision.append(rtn)
        elif sig == -1:
            if rtn < 0:
                tn += 1
                gain.append(-rtn)
            else:
                fn += 1
                loss.append(-rtn)

    accuracy = 0
    precision = 0
    recall = 0
    ev = 0
    if tp + fp + tn + fn == 0 or tp + fp == 0 or tp + fn == 0:
        accuracy = 0
        precision = 0
        recall = 0
        ev = 0
        precision_ev = 0
    elif len(gain) == 0 or len(loss) == 0:
        accuracy = (tp + tn) / (tp + fp + tn + fn)
        precision = tp / (tp + fp)
        recall = tp / (tp + fn)
        ev = 0
        if len(gain_precision) == 0 or len(loss_precision) == 0:
            precision_ev = 0
        else:
            precision_ev = (sum(gain_precision) / len(gain_precision))*precision + (sum(loss_precision) / len(loss_precision))*(1-precision)

    else:
        accuracy = (tp + tn) / (tp + fp + tn + fn)
        precision = tp / (tp + fp)
        recall = tp / (tp + fn)
        ev = (sum(gain) / len(gain))*accuracy + (sum(loss) / len(loss))*(1-accuracy)
        if len(gain_precision) == 0 or len(loss_precision) == 0:
            precision_ev = 0
        else:
            precision_ev = (sum(gain_precision) / len(gain_precision))*precision + (sum(loss_precision) / len(loss_precision))*(1-precision)

    result = {
        'tp' : tp,
        'fp' : fp,
        'tn' : tn,
        'fn' : fn,
        'avail_count' : tp + fp + tn + fn,
        'ttl_count' : ttl_count,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'ev': ev,
        'precision_ev' : precision_ev,
    }

    return result

In [20]:
with open(path_out + "1.json", 'r') as f:
    res = json.load(f)

result = eval(price, res)

print(f'''tp:{result["tp"]}, fp:{result["fp"]}, tn:{result["tn"]}, fn:{result["fn"]}, Available ask:[{result["avail_count"]}/{result["ttl_count"]}],
Accuracy : {round(result["accuracy"],3) }, Expected Value : {round(result["ev"], 7)},
Precision : {round(result["precision"],5)}, Precision Expected Value : {round(result["precision_ev"], 7)},
Recall : {round(result["recall"],5)}''')  

tp:161, fp:158, tn:77, fn:77, Available ask:[473/474],
Accuracy : 0.503, Expected Value : 0.0010672,
Precision : 0.5047, Precision Expected Value : 0.0012941,
Recall : 0.67647
